# 03 — RAG Evaluation

**Goal**: Measure how well your RAG system performs.

Without evaluation, you don't know if your RAG improvements actually work.

In [ ]:
# !pip install ragas

import requests
import numpy as np

def ollama_generate(prompt):
    resp = requests.post("http://localhost:11434/api/generate", json={
        "model": "llama3.2",
        "prompt": prompt,
        "stream": False,
        "temperature": 0.1
    })
    return resp.json()["response"]

print("Ready")

## 1. What to Evaluate?

### Retrieval Quality
- **Precision**: Are retrieved docs relevant?
- **Recall**: Did we retrieve all relevant docs?
- **Hit Rate**: Does the relevant doc appear in top-k?

### Generation Quality
- **Faithfulness**: Does the answer stick to the context?
- **Relevance**: Does the answer address the question?
- **Helpfulness**: Is the answer useful?

### End-to-End
- **RAGAS** (RAG Assessment) framework

## 2. Building a Test Dataset

We need: questions, ground-truth answers, and context documents.

In [ ]:
# Our test set
test_data = [
    {
        "question": "What is the Transformer architecture based on?",
        "ground_truth": "Self-attention mechanisms",
        "context": "The Transformer architecture, introduced in 'Attention is All You Need', uses self-attention mechanisms to process all tokens in parallel."
    },
    {
        "question": "What does LoRA stand for and what does it do?",
        "ground_truth": "Low-Rank Adaptation. It adds trainable rank decomposition matrices to attention layers, reducing trainable parameters.",
        "context": "LoRA (Low-Rank Adaptation) adds trainable rank decomposition matrices to attention layers, reducing trainable parameters by 99%."
    },
    {
        "question": "How does RAG reduce hallucinations?",
        "ground_truth": "RAG provides relevant context documents to the LLM, grounding its answers in retrieved information.",
        "context": "RAG reduces hallucinations by providing relevant context documents to ground the LLM's generated answers in retrieved information."
    },
]

print(f"Test set: {len(test_data)} Q&A pairs")

## 3. Manual Evaluation Functions

Let's build our own simple evaluators.

In [ ]:
def faithfulness_score(answer, context):
    """Ask LLM to evaluate faithfulness (1-5)."""
    prompt = f"""Rate how faithful this answer is to the context (1-5, integer only):

Context: {context}

Answer: {answer}

Faithfulness score (1=hallucination, 5=fully grounded):"""
    score = ollama_generate(prompt).strip()
    try:
        return int(score[0])
    except:
        return 3

def answer_relevance(question, answer):
    """Ask LLM to rate relevance (1-5)."""
    prompt = f"""Rate how well this answer addresses the question (1-5, integer only):

Question: {question}
Answer: {answer}

Relevance score (1=irrelevant, 5=perfect):"""
    score = ollama_generate(prompt).strip()
    try:
        return int(score[0])
    except:
        return 3

print("Evaluator functions ready")

## 4. Running Evaluation

Let's test our simple RAG system and score it.

In [ ]:
def simple_rag(question, context):
    """Minimum viable RAG."""
    prompt = f"""Answer using the context.

Context: {context}

Question: {question}
Answer:"""
    return ollama_generate(prompt)

results = []
for item in test_data:
    answer = simple_rag(item["question"], item["context"])
    faithful = faithfulness_score(answer, item["context"])
    relevant = answer_relevance(item["question"], answer)
    
    results.append({
        "question": item["question"],
        "answer": answer[:100],
        "faithfulness": faithful,
        "relevance": relevant
    })
    
    print(f"\nQ: {item['question']}")
    print(f"A: {answer[:80]}...")
    print(f"Faithfulness: {faithful}/5, Relevance: {relevant}/5")

avg_faithful = np.mean([r['faithfulness'] for r in results])
avg_relevant = np.mean([r['relevance'] for r in results])
print(f"\n{'='*40}")
print(f"AVERAGE: Faithfulness={avg_faithful:.1f}/5, Relevance={avg_relevant:.1f}/5")

## 5. RAGAS Metrics (Conceptual)

The RAGAS framework formalizes evaluation into 5 metrics:

| Metric | What It Measures | How |
|---|---|---|
| Faithfulness | Does answer match context? | LLM judges each claim |
| Answer Relevance | Does answer address question? | Cosine similarity with generated questions |
| Context Precision | Are retrieved docs all relevant? | Ranking of relevant docs |
| Context Recall | Did we miss relevant docs? | Ground truth coverage |
| Answer Correctness | Is the answer factually right? | LLM comparison |

To use RAGAS:
```python
from ragas import evaluate
from datasets import Dataset

dataset = Dataset.from_dict({
    "question": [...],
    "answer": [...],
    "contexts": [[...], ...],
    "ground_truth": [...]
})
scores = evaluate(dataset)
```

Note: RAGAS requires OpenAI by default — for local evaluation, use our manual approach above.

## 6. A/B Testing RAG Strategies

Use evaluation to compare approaches.

In [ ]:
def evaluate_strategy(strategy_name, rag_function, test_data):
    """Evaluate a RAG strategy across test data."""
    scores = []
    for item in test_data:
        answer = rag_function(item["question"], item["context"])
        faithful = faithfulness_score(answer, item["context"])
        scores.append(faithful)
    return np.mean(scores)

# Example: compare with and without system prompt
def rag_with_system(question, context):
    prompt = f"""You are a precise AI assistant. Only use the context to answer.
If unsure, say you don't know.

Context: {context}
Question: {question}
Answer:"""
    return ollama_generate(prompt)

score_basic = evaluate_strategy("basic", simple_rag, test_data[:1])
score_system = evaluate_strategy("with system prompt", rag_with_system, test_data[:1])
print(f"Basic RAG faithfulness: {score_basic}")
print(f"With system prompt: {score_system}")

## Key Takeaways

1. **Evaluate retrieval** (precision, recall) **and** generation (faithfulness, relevance)
2. **Build a test set** of Q&A pairs with ground truth
3. **Use LLM-as-judge** when you don't have ground truth
4. **RAGAS** is the standard framework (but often needs OpenAI)
5. **A/B test** your RAG improvements — measure, don't guess!

**Next**: Fine-tuning — adapt models to your specific needs →